#Classification (ResNet-50 / TResNet-L — Multi-label Classification)

In [ ]:
pip install roboflow torch torchvision timm scikit-learn

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
dataset = project.version(1).download("folder")  # use 'folder' for classification

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# --- Parameters ---
NUM_CLASSES = 17        # adjust to your dataset
BATCH_SIZE  = 32
EPOCHS      = 50
LR          = 1e-4
IMG_SIZE    = 224
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Transforms ---
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# --- Datasets ---
train_ds = datasets.ImageFolder(f"{dataset.location}/train", transform=transform)
val_ds   = datasets.ImageFolder(f"{dataset.location}/valid", transform=transform)
test_ds  = datasets.ImageFolder(f"{dataset.location}/test",  transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

# --- Model: ResNet-50 with ImageNet weights ---
model = models.resnet50(weights="IMAGENET1K_V2")
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)  # replace head
model = model.to(DEVICE)

# Swap to TResNet-L: pip install timm
# import timm
# model = timm.create_model("tresnet_l", pretrained=True, num_classes=NUM_CLASSES)

# --- Optimizer & loss ---
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()   # use BCEWithLogitsLoss for multi-label

# --- Training loop ---
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    correct = 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = correct / len(train_ds)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss:.3f} | Train Acc: {train_acc:.4f}")

torch.save(model.state_dict(), "classifier.pth")

In [ ]:
from sklearn.metrics import classification_report, f1_score

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(DEVICE))
        preds = outputs.argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds,
      target_names=train_ds.classes))
print("F1 (weighted):", f1_score(all_labels, all_preds, average="weighted"))

#Object Detection (YOLOv26)

In [2]:
pip install ultralytics

Training

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 pre-trained on COCO
model = YOLO("yolov8m.pt")   # n / s / m / l / x — m is a good default

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=150,
    imgsz=640,
    batch=16,
    lr0=1e-3,              # initial LR
    lrf=0.01,              # final LR factor
    momentum=0.937,        # SGD momentum
    weight_decay=5e-4,
    optimizer="SGD",
    augment=True,          # mosaic, flip, scale augmentation
    patience=20,           # early stopping
    project="sewer_detection",
    name="yolov8_run1"
)

evaluate on test set

In [ ]:
metrics = model.val(
    data=f"{dataset.location}/data.yaml",
    split="test"
)

print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

run inference

In [ ]:
results = model.predict(
    source="path/to/test/image.jpg",
    conf=0.25,
    save=True
)

for r in results:
    print(r.boxes.xyxy)   # bounding boxes → pass to SAM

#SAM (ViT-H) — Image Segmentation
SAM takes YOLO bounding boxes as prompts. Run detection first, then pipe boxes into SAM.

In [ ]:
pip install opencv-python
pip install git+https://github.com/facebookresearch/segment-anything.git
# Download checkpoint:
# wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

yolo + sam pipeline

In [ ]:
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from segment_anything import sam_model_registry, SamPredictor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load models ---
yolo = YOLO("sewer_detection/yolov8_run1/weights/best.pt")

sam = sam_model_registry["vit_h"](checkpoint="sam_vit_h_4b8939.pth")
sam.to(DEVICE)
predictor = SamPredictor(sam)

# --- Inference function ---
def segment_defects(image_path, conf_thresh=0.25):
    image_bgr = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    # Step 1: YOLO detection
    results = yolo(image_bgr, conf=conf_thresh)[0]
    boxes = results.boxes.xyxy.cpu().numpy()   # [x1, y1, x2, y2]
    labels = results.boxes.cls.cpu().numpy()

    if len(boxes) == 0:
        print("No defects detected.")
        return []

    # Step 2: Set image in SAM
    predictor.set_image(image_rgb)

    defects = []
    for box, label in zip(boxes, labels):
        masks, scores, _ = predictor.predict(
            box=box,
            multimask_output=True
        )
        best_idx  = np.argmax(scores)
        best_mask = masks[best_idx]

        defects.append({
            "class":      int(label),
            "bbox":       box,
            "mask":       best_mask,
            "area_px":    int(best_mask.sum()),
            "confidence": float(scores[best_idx])
        })

    return defects

# --- Run & visualise ---
defects = segment_defects("path/to/test/image.jpg")
for d in defects:
    print(f"Class {d['class']} | Area: {d['area_px']}px | Score: {d['confidence']:.3f}")

IOU evaluation (segmentation quality)

In [ ]:
def compute_iou(pred_mask, gt_mask):
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union        = np.logical_or(pred_mask,  gt_mask).sum()
    return intersection / union if union > 0 else 0.0

# Compare SAM masks against ground-truth annotations
ious = [compute_iou(d["mask"], gt) for d, gt in zip(defects, ground_truths)]
print(f"Mean IoU: {np.mean(ious):.4f}")